<div style="display:flex; justify-content:space-between; align-items:center; width:100%; margin:8px 0 24px 0;">
  <div style="text-align:left;">
    <img src="https://www.ec-nantes.fr/medias/photo/logocn-rvb_1648479844750-png?ID_FICHE=178994&amp;INLINE=FALSE" alt="Centrale Nantes" style="height:72px; width:auto;">
  </div>
  <div style="text-align:right; font-size:18px; font-weight:600; color:#17324d; line-height:1.35;">
    MSc. CORO DASSIP
  </div>
</div>

<div style="border:2px solid #333; padding:14px 20px; margin:15px auto 25px auto; width:85%; max-width:900px; box-sizing:border-box; text-align:center;">
  <h1 style="margin:0;"><b>Camera Calibration</b></h1>
</div>

## Context

Camera calibration estimates the geometric relationship between a known calibration target and its image observations. This project uses multiple views of one planar chessboard to estimate a pinhole camera model and evaluate its geometric consistency.

## Problem Statement

Given multiple images of the same planar chessboard, solve the complete calibration pipeline required by Tasks 1–13 below.

The final solution must estimate the intrinsic matrix $K$, recover one pose $(R,t)$ for every valid image, reproject the known calibration points, quantify reprojection error, generate the required figures, and pass the numerical/output validation checks.

Lens-distortion estimation and nonlinear refinement are outside the implemented scope.

## Inputs and Fixed Parameters

| Item | Value |
| --- | ---: |
| Input folder | `../data/calibration_images/` |
| Input extension | `*.jpg` |
| Internal corners | $8 \times 6$ |
| Square size | $0.03\,\mathrm{m}$ |
| Minimum valid views | 3 |
| Calibration plane | $Z=0$ |
| Error unit | pixels |

## 1. Load the Sorted JPEG Calibration Images

**Required result:** discover all calibration JPEG files from the repository-relative input folder, sort them deterministically by filename, and stop with a clear error if the folder or files are missing.

## 2. Detect and Refine Chessboard Corners

**Required result:** detect the complete $8 \times 6$ internal-corner pattern in each usable image and refine the detected corner coordinates to sub-pixel precision. Views with failed detections must be rejected explicitly.

## 3. Build the Planar World Coordinates

**Required result:** generate the 48 planar calibration coordinates in metres, with the first internal corner at the origin, $X$ along the 8-corner direction, $Y$ along the 6-corner direction, and $Z=0$.

## 4. Compute $T_{\mathrm{image}}$ and $T_{\mathrm{plane}}$

**Required result:** normalize the image points and planar points independently so that each set has zero centroid and mean distance $\sqrt{2}$ from the origin.

## 5. Build the DLT Matrix $Q$ and Solve $Q\mathbf{h}=0$ by SVD

**Required result:** for every valid view, build the normalized DLT system from all point correspondences and recover the normalized homography parameters from the right-singular vector associated with the smallest singular value.

## 6. Denormalize Each Homography

**Required result:** recover the image-space homography using

$$
H=T_{\mathrm{image}}^{-1}H_nT_{\mathrm{plane}}
$$

and normalize the arbitrary scale so that $H_{33}=1$.

## 7. Build the Zhang Matrix $V$ and Solve $Vb=0$ by SVD

**Required result:** form the two Zhang constraints from every valid homography, stack them into $V$, solve $Vb=0$ by SVD, and retain the homogeneous solution vector $b$.

## 8. Recover $\alpha,\beta,\gamma,u_0,v_0$ and Construct $K$

**Required result:** recover the intrinsic parameters $\alpha,\beta,\gamma,u_0,v_0$ from $b$ and assemble

$$
K=
\begin{bmatrix}
\alpha&\gamma&u_0\\
0&\beta&v_0\\
0&0&1
\end{bmatrix}.
$$

## 9. Recover $R$ and $t$ for Every Retained View

**Required result:** recover one rotation matrix $R$ and one translation vector $t$ from $K$ and $H$ for every retained view. The final rotation must be orthonormal with determinant $+1$.

## 10. Reproject the $Z=0$ Calibration Points

**Required result:** embed each planar calibration point as $[X,Y,0]^T$, transform it into the camera frame using $(R,t)$, project it through $K$, and convert the homogeneous image coordinates to pixels.

## 11. Compute Point-wise Errors, Mean Error and RMSE

**Required result:** compute the Euclidean pixel error for every calibration corner, then report per-view mean error, per-view RMSE, overall mean error and overall RMSE.

## 12. Produce and Save the Six Required Diagnostic Figures

The implementation must save exactly these six diagnostic figures:

1. `detected_chessboard_corners.png`
2. `homography_estimation_pipeline.png`
3. `estimated_camera_poses.png`
4. `reprojection_results.png`
5. `mean_reprojection_error_by_view.png`
6. `reprojection_error_distribution.png`

## 13. Run the Numerical and Output-file Validation Checks

The completed solution must validate:

- finite $3\times3$ intrinsic matrix $K$;
- 48 image and planar points for every retained view;
- finite homographies;
- $R^TR\approx I$ for every recovered rotation;
- $\det(R)\approx1$;
- finite reprojection errors;
- presence of all six required output figures.

## Completion Criterion

The problem is fully solved only when **Tasks 1–13 are all implemented and validated** in `camera_calibration.ipynb` using the same notation and ordering defined here.